Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

/home/jokello/ML/IntelliScore/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Dataset

In [2]:
data = pd.read_csv('../data/customers_transactions_features.csv')

Basic info

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 40 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   CustomerID            500000 non-null  object 
 1   Age                   500000 non-null  int64  
 2   Gender                500000 non-null  object 
 3   MaritalStatus         500000 non-null  object 
 4   Education             500000 non-null  object 
 5   EmploymentStatus      500000 non-null  object 
 6   Income                500000 non-null  int64  
 7   LoanAmount            500000 non-null  int64  
 8   NumLoans              500000 non-null  int64  
 9   CreditHistoryLength   500000 non-null  int64  
 10  AvgMonthlyExpenses    500000 non-null  int64  
 11  Savings               500000 non-null  int64  
 12  CreditCardUsage       500000 non-null  object 
 13  MobileMoneyUsage      500000 non-null  object 
 14  LatePayments          500000 non-null  int64  
 15  

Descriptive Analysis

In [4]:
data.describe(include='all')

,CustomerID,Age,Gender,MaritalStatus,Education,EmploymentStatus,Income,LoanAmount,NumLoans,CreditHistoryLength,...,FraudRate,ChannelRatio_ATM,ChannelRatio_Mobile,ChannelRatio_Online,ChannelRatio_POS,TimeRatio_Afternoon,TimeRatio_Evening,TimeRatio_Morning,TimeRatio_Night,WeekendActivityRatio
count,500000,500000.000000,500000,500000,500000,500000,500000.000000,500000.000000,500000.000000,500000.000000,...,315680.000000,315680.000000,315680.000000,315680.000000,315680.000000,315680.000000,315680.000000,315680.000000,315680.000000,315680.000000
unique,500000,NaN,2,4,4,4,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,CUST499961,NaN,Female,Single,Secondary,Employed,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,260178,225593,200334,300094,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,33.539122,NaN,NaN,NaN,NaN,23158.413736,6881.590156,5.344982,11.951502,...,0.129996,0.100347,0.199651,0.249700,0.450301,0.350914,0.250156,0.249606,0.149325,0.285600
std,NaN,10.061778,NaN,NaN,NaN,NaN,10651.480490,4169.117214,3.296935,7.542322,...,0.294153,0.263204,0.349997,0.378976,0.435509,0.417767,0.379152,0.378824,0.311855,0.395328
min,NaN,18.000000,NaN,NaN,NaN,NaN,2180.000000,500.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,26.000000,NaN,NaN,NaN,NaN,15708.000000,3958.000000,2.000000,5.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,NaN,34.000000,NaN,NaN,NaN,NaN,20736.000000,5715.000000,5.000000,12.000000,...,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,NaN,41.000000,NaN,NaN,NaN,NaN,29114.250000,8993.000000,8.000000,19.000000,...,0.000000,0.000000,0.333333,0.500000,1.000000,0.666667,0.500000,0.500000,0.000000,0.500000


Handle Null Values

In [5]:
if data.isnull().sum().sum() == 0:
    print("No null values found.")
else:
    print("Null values found:")
    print(data.isnull().sum())
    data.fillna(0, inplace=True)
    print("Null values filled with 0.")

Null values found:
CustomerID                   0
Age                          0
Gender                       0
MaritalStatus                0
Education                    0
EmploymentStatus             0
Income                       0
LoanAmount                   0
NumLoans                     0
CreditHistoryLength          0
AvgMonthlyExpenses           0
Savings                      0
CreditCardUsage              0
MobileMoneyUsage             0
LatePayments                 0
Default                      0
DebtToIncome                 0
ExpenseRatio                 0
SavingsRate                  0
LoansPerYearHistory          0
HighDebtFlag                 0
LowSavingsFlag               0
FrequentLatePayer            0
CreditCardUsageEnc           0
MobileMoneyUsageEnc     500000
TotalTransactions       184320
AvgTransactionAmount    184320
MaxTransactionAmount    184320
TransactionStdDev       367495
TotalFrauds             184320
FraudRate               184320
ChannelRatio_ATM    

In [6]:
data[data['FraudRate'] == 1]

,CustomerID,Age,Gender,MaritalStatus,Education,EmploymentStatus,Income,LoanAmount,NumLoans,CreditHistoryLength,...,FraudRate,ChannelRatio_ATM,ChannelRatio_Mobile,ChannelRatio_Online,ChannelRatio_POS,TimeRatio_Afternoon,TimeRatio_Evening,TimeRatio_Morning,TimeRatio_Night,WeekendActivityRatio
10,CUST000011,30,Male,Married,Primary,Employed,23219,4343,5,22,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
40,CUST000041,42,Male,Married,Secondary,Employed,45994,5385,10,4,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
43,CUST000044,18,Female,Single,Secondary,Self-employed,15708,4574,10,4,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
48,CUST000049,38,Female,Married,Postgraduate,Employed,16844,7681,6,22,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
74,CUST000075,18,Male,Married,Secondary,Employed,12774,5695,0,2,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499867,CUST499868,43,Female,Married,Primary,Unemployed,10369,2679,0,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
499892,CUST499893,35,Female,Single,Secondary,Employed,35211,7678,0,18,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
499931,CUST499932,22,Female,Divorced,Postgraduate,Employed,30359,6686,8,10,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
499950,CUST499951,40,Female,Married,Primary,Self-employed,25948,8384,0,11,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


TRAINING PIPELINE

In [ ]:
# Encoding categorical features
categorical_features = data.select_dtypes(include=['object']).columns
for col in categorical_features:
    if data[col].nunique() <= 10:
        encoder = OneHotEncoder(sparse_output=False)
        encoded = encoder.fit_transform(data[[col]])
        encoded_df = pd.DataFrame(encoded, columns=[f"{col}_{cat}" for cat in encoder.categories_[0]])
        data = pd.concat([data.drop(columns=[col]), encoded_df], axis=1)
    else:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])

# Credit scoring
target_credit = "Default"
X_credit = data.drop(columns=[target_credit, "FraudRate", 'CustomerID'])
y_credit = data[target_credit]

# Fraud detection
target_fraud = "FraudRate"
X_fraud = data.drop(columns=[target_fraud, target_credit, 'CustomerID'])
y_fraud = data[target_fraud]

# Binarize FraudRate: 0.0 -> 0 (no fraud), >0.0 -> 1 (fraud)
y_fraud = (y_fraud > 0.0).astype(int)

# Remove classes in y_fraud with less than 2 samples (not strictly needed now, but kept for consistency)
fraud_counts = y_fraud.value_counts()
valid_classes = fraud_counts[fraud_counts >= 2].index
fraud_mask = y_fraud.isin(valid_classes)
X_fraud_valid = X_fraud[fraud_mask]
y_fraud_valid = y_fraud[fraud_mask]

# Split datasets
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_credit, y_credit, test_size=0.2, random_state=42, stratify=y_credit)
Xf_train, Xf_test, yf_train, yf_test = train_test_split(X_fraud_valid, y_fraud_valid, test_size=0.2, random_state=42, stratify=y_fraud_valid)

# Standardize for Logistic Regression
scaler = StandardScaler()
Xc_train_scaled = scaler.fit_transform(Xc_train)
Xc_test_scaled = scaler.transform(Xc_test)

# Save scaler for future use
joblib.dump(scaler, "credit_scaler.pkl")

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", use_label_encoder=False)
}

# Training & evaluating the models
def evaluate_models(X_train, X_test, y_train, y_test, task_name):
    results = []
    trained_models = {}

    # Check if there are at least two classes in y_train and y_test
    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        print(f"Warning: {task_name} - Only one class present in training or test data. Skipping model training.")
        return results, trained_models

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, y_prob)

        results.append([task_name, name, acc, prec, rec, f1, auc])
        trained_models[name] = model

        # ROC curve
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.2f})")

    plt.plot([0, 1], [0, 1], '--', color='gray')
    plt.title(f"ROC Curve - {task_name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()

    return results, trained_models

# Evaluate Credit Scoring
credit_results, credit_models = evaluate_models(Xc_train_scaled, Xc_test_scaled, yc_train, yc_test, "Credit Scoring")

# Evaluate Fraud Detection
fraud_results, fraud_models = evaluate_models(Xf_train, Xf_test, yf_train, yf_test, "Fraud Detection")

# Combine results
results_df = pd.DataFrame(credit_results + fraud_results, columns=["Task", "Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"])
print(results_df)

# Select best models by ROC-AUC
best_models = results_df.loc[results_df.groupby("Task")["ROC-AUC"].idxmax()]
print("\nBest models selected:")
print(best_models)

# Save best models
if not best_models.empty:
    for _, row in best_models.iterrows():
        task = row["Task"]
        model_name = row["Model"]
        if task.startswith("Credit"):
            best_model = credit_models[model_name]
        else:
            best_model = fraud_models[model_name]
        filename = f"best_{task.replace(' ', '_').lower()}_{model_name.lower().replace(' ', '_')}.pkl"
        joblib.dump(best_model, filename)
        print(f"Saved: {filename}")

# Precision-Recall curve for fraud detection
for name, model in fraud_models.items():
    y_prob = model.predict_proba(Xf_test)[:, 1]
    prec, rec, _ = precision_recall_curve(yf_test, y_prob)
    plt.plot(rec, prec, label=f"{name}")

plt.title("Precision-Recall Curve - Fraud Detection")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.show()

# Feature importance (XGBoost - Fraud)
xgb_fraud = fraud_models.get("XGBoost")
importances = xgb_fraud.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices][:20], y=X_fraud.columns[indices][:20])
plt.title("Top 20 Feature Importances (Fraud Detection)")
plt.show()

# SHAP values (Fraud)
explainer = shap.TreeExplainer(xgb_fraud)
shap_values = explainer.shap_values(Xf_test[:200])
shap.summary_plot(shap_values, Xf_test[:200])

PREDICTION

In [ ]:
def score_new_data(new_data_csv):
    """Load new customer/transaction data and score using best saved models."""

    # Load new data
    new_data = pd.read_csv(new_data_csv)

    # Load scaler
    scaler = joblib.load("credit_scaler.pkl")

    # Load best models
    credit_model = joblib.load("best_credit_scoring_xgboost.pkl") if os.path.exists("best_credit_scoring_xgboost.pkl") else None
    fraud_model = joblib.load("best_fraud_detection_xgboost.pkl") if os.path.exists("best_fraud_detection_xgboost.pkl") else None

    results = {}

    if credit_model:
        X_new_credit = new_data.drop(columns=["FraudRate"], errors="ignore")
        X_new_credit_scaled = scaler.transform(X_new_credit)
        new_data["CreditRisk"] = credit_model.predict_proba(X_new_credit_scaled)[:, 1]
        results["CreditRisk"] = new_data[["CreditRisk"]]

    if fraud_model:
        X_new_fraud = new_data.drop(columns=["Defaulted"], errors="ignore")
        new_data["FraudProbability"] = fraud_model.predict_proba(X_new_fraud)[:, 1]
        results["FraudProbability"] = new_data[["FraudProbability"]]

    # Save results
    new_data.to_csv("scored_new_data.csv", index=False)
    print("Scoring complete and saved to 'scored_new_data.csv'")
    return results

# Example usage
# scored_results = score_new_data("../data/new_customers_transactions.csv")
